In [38]:
import os
import gradio as gr
import requests
import json
from dotenv import load_dotenv
from openai import OpenAI
from scrapper import fetch_website_links,fetch_selenium_website_contents, fetch_website_contents




In [39]:
load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
ollama_api_key = ollama_key = os.getenv("OLLAMA_API_KEY")


if google_api_key:
	print("Google API key exists")
else:
	print("Google API key not set")	

if groq_api_key:
	print("Groq API key exists")
else:
	print("Groq API key not set")	

if ollama_api_key:
	print("Ollama API key exists")
else:
	print("Ollama API key not set")	


Google API key exists
Groq API key exists
Ollama API key exists


In [40]:
openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key=ollama_api_key, base_url=ollama_url)

In [41]:
system_message= "Youa are a helpful assistant that responds in markdown without code blocks"

In [42]:
system_prompt= """
		You are provided with a list of links found on a webpage.
		You are able to decide which of the links would be most relevant to include in a brochure about the company,
		such as links to an About page, or a Company page, or Careers/Jobs pages.
		You should respond in JSON as in this example:

		{
			"links": [
				{"type": "about page", "url": "https://full.url/goes/here/about"},
				{"type": "careers page", "url": "https://another.full.url/careers"}
			]
		}
		"""

In [43]:
def get_user_prompt_links(url):
	user_prompt = f"""
		Here is the list of links on the website {url} -
		Please decide which of these are relevant web links for a brochure about the company, 
		respond with the full https URL in JSON format.
		Do not include Terms of Service, Privacy, email links.

		Links (some might be relative links):
		"""
	links = fetch_website_links(url)
	user_prompt += "\n".join(links)
	return user_prompt


In [76]:
def chunk_streams(stream):
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [45]:
def get_relevant_links(url):

	response = ollama.chat.completions.create(
		model= "llama3.2",
		messages= [
			{"role": "system", "content": system_prompt},
			{"role": "user", "content": get_user_prompt_links(url)}
		],
		response_format={"type": "json_object"}
	)

	results =  response.choices[0].message.content
	links = json.loads(results)
	return links

In [ ]:
def fetch_pages_and_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = get_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant links:\n"
    for link in relevant_links["links"]:
        try:
            page = fetch_website_contents(link["url"])
        except requests.exceptions.RequestException as e:
            print(f"Skipping {link['url']}: {e.__class__.__name__}")
            continue
        result += f"\n\n### Link: {link['type']}\n{page}"
    return result
  

In [71]:
bronchure_system_prompt= """
	You are an assistant that analyses several pages contents and links of a company website and create a short bronchure of company prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information. 
"""

In [70]:
def get_bronchure_user_prompt(company_name, url):
	user_prompt = f"""
		You are looking at a company called: {company_name}
		Here are the contents of its landing page and other relevant pages;
		use this information to build a short brochure of the company in markdown without code blocks.\n\n
		"""
	user_prompt += fetch_pages_and_relevant_links(url)
	user_prompt = user_prompt[:5000]
	return user_prompt

	

In [68]:
def stream_gpt(company_name, url):
	bronchure_user_prompt = get_bronchure_user_prompt(company_name, url)
	messages=[
			{"role": "system", "content": bronchure_system_prompt},
			{"role": "user", "content": bronchure_user_prompt},
			]
	stream = groq.chat.completions.create(model="openai/gpt-oss-120b", messages=messages, stream=True)
	yield from chunk_streams(stream)

In [69]:
def stream_ollama(company_name, url):
	bronchure_user_prompt = get_bronchure_user_prompt(company_name, url)
	messages=[
			{"role": "system", "content": bronchure_system_prompt},
			{"role": "user", "content": bronchure_user_prompt}
			]
	stream = ollama.chat.completions.create(model="llama3.2", messages=messages, stream=True)
	yield from chunk_streams(stream)

In [ ]:
def stream_model(company, url, model):
	if model == "GPT":
		result = stream_gpt(company, url)
	elif model == "OLLAMA":
		result = stream_ollama(company, url)
	else:
		raise ValueError("No such model exist")
	yield from result

In [ ]:
company_name_input = gr.Textbox(label="Company Name", info="Enter Company Name")
company_url_input = gr.Textbox(label="Company Website", info="Enter Company website starting with http or https")
model_selector = gr.Dropdown(["GPT", "OLLAMA"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
	fn=stream_model, title="Company Brochure Generator",
	inputs=[company_name_input, company_url_input, model_selector],
	outputs=[message_output],
	examples=[
		["Andela", "https://www.andela.com", "GPT"],
		["Hugging Face", "https://huggingface.co", "OLLAMA"],
		],
	flagging_mode="never"

)

view.launch()



* Running on local URL:  http://127.0.0.1:7891
* To create a public link, set `share=True` in `launch()`.


=-=-=-=-=-==->>>>> Andela https://www.andela.com GPT
=-=-=-=-=-==->>>>> Irembo https://www.irembo.com GPT
